# Homomorphic Encryption Attack and Privacy Analysis

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.stats import entropy
import hashlib
from collections import defaultdict

class HomomorphicEncryptionAnalyzer:
    def __init__(self, lattice_dim=512, modulus=2**14):
        self.lattice_dim = lattice_dim
        self.modulus = modulus
        self.noise_bound = int(np.sqrt(lattice_dim))
        self.secret_key = np.random.randint(-1, 2, lattice_dim)
        self.public_key = self.generate_public_key()
        self.attack_vectors = defaultdict(list)
        self.scaler = StandardScaler()

    def generate_public_key(self):
        A = np.random.randint(0, self.modulus, (self.lattice_dim, self.lattice_dim))
        e = np.random.randint(-self.noise_bound, self.noise_bound + 1, self.lattice_dim)
        b = (-np.dot(A, self.secret_key) + e) % self.modulus
        return (A, b)

    def encrypt(self, plaintext):
        A, b = self.public_key
        r = np.random.randint(0, 2, self.lattice_dim)
        noise = np.random.randint(-2, 3, 1)[0]
        c0 = (np.dot(r, A) % self.modulus).astype(int)
        c1 = (np.dot(r, b) + plaintext + noise) % self.modulus
        return (c0, c1)

    def decrypt(self, ciphertext):
        c0, c1 = ciphertext
        decrypted = (c1 - np.dot(c0, self.secret_key)) % self.modulus
        return decrypted if decrypted < self.modulus // 2 else decrypted - self.modulus

    def homomorphic_add(self, ct1, ct2):
        return ((ct1[0] + ct2[0]) % self.modulus, (ct1[1] + ct2[1]) % self.modulus)

    def homomorphic_multiply(self, ct1, ct2):
        c0_new = (ct1[0] * ct2[0]) % self.modulus
        c1_new = (ct1[0] * ct2[1] + ct1[1] * ct2[0] + ct1[1] * ct2[1]) % self.modulus
        return (c0_new, c1_new)

    def analyze_noise_growth(self, operations):
        initial_noise = self.noise_bound
        current_noise = initial_noise
        noise_trajectory = [current_noise]

        for op in operations:
            if op == 'add':
                current_noise = current_noise * 2
            elif op == 'multiply':
                current_noise = current_noise ** 2
            noise_trajectory.append(min(current_noise, self.modulus // 4))

        max_depth = len([n for n in noise_trajectory if n < self.modulus // 4])
        return {'max_depth': max_depth, 'final_noise': noise_trajectory[-1], 'trajectory': noise_trajectory}

    def differential_privacy_attack(self, encrypted_dataset, epsilon=1.0):
        attack_success = []
        for i, ciphertext in enumerate(encrypted_dataset):
            neighboring_queries = []
            for j in range(min(10, len(encrypted_dataset))):
                if i != j:
                    difference = self.homomorphic_add(ciphertext, self.encrypt(-self.decrypt(encrypted_dataset[j])))
                    neighboring_queries.append(abs(self.decrypt(difference)))

            if neighboring_queries:
                sensitivity = max(neighboring_queries)
                privacy_loss = sensitivity / epsilon if epsilon > 0 else float('inf')
                attack_success.append(privacy_loss > 1.0)

        return np.mean(attack_success) if attack_success else 0

    def side_channel_analysis(self, computation_traces):
        features = []
        for trace in computation_traces:
            power_variance = np.var(trace)
            spectral_entropy = entropy(np.abs(np.fft.fft(trace)))
            peak_frequency = np.argmax(np.abs(np.fft.fft(trace)))
            correlation = np.corrcoef(trace[:-1], trace[1:])[0, 1] if len(trace) > 1 else 0
            features.append([power_variance, spectral_entropy, peak_frequency, correlation])

        if len(features) > 1:
            kmeans = KMeans(n_clusters=min(3, len(features)), random_state=42)
            clusters = kmeans.fit_predict(self.scaler.fit_transform(features))
            distinguishability = len(set(clusters)) / len(clusters)
            return distinguishability > 0.5
        return False

    def lattice_reduction_attack(self, public_samples=100):
        A, b = self.public_key
        sample_matrix = []

        for _ in range(min(public_samples, self.lattice_dim)):
            r = np.random.randint(0, 2, self.lattice_dim)
            sample = (np.dot(r, A) % self.modulus, np.dot(r, b) % self.modulus)
            sample_matrix.append(list(sample[0]) + [sample[1]])

        if not sample_matrix: return {'success': False, 'complexity': float('inf')}

        sample_matrix = np.array(sample_matrix)
        condition_number = np.linalg.cond(sample_matrix) if sample_matrix.shape[0] == sample_matrix.shape[1] else 1
        attack_complexity = condition_number * (self.modulus ** 0.5)

        return {'success': attack_complexity < 2**40, 'complexity': attack_complexity}

    def comprehensive_security_analysis(self, operations, encrypted_data, computation_traces):
        noise_analysis = self.analyze_noise_growth(operations)
        dp_vulnerability = self.differential_privacy_attack(encrypted_data)
        side_channel_leak = self.side_channel_analysis(computation_traces)
        lattice_attack = self.lattice_reduction_attack()

        security_score = 1.0
        if noise_analysis['max_depth'] < 5: security_score -= 0.3
        if dp_vulnerability > 0.7: security_score -= 0.3
        if side_channel_leak: security_score -= 0.2
        if lattice_attack['success']: security_score -= 0.4

        return {'score': max(security_score, 0), 'noise': noise_analysis, 'dp_attack': dp_vulnerability,
                'side_channel': side_channel_leak, 'lattice': lattice_attack}